# Démo 2 : manipuler des tenseurs

Objectif : manipuler la structure de données de PyTorch, le tenseur. Des températures, une petite image, un tableau de ventes : on crée, on lit, on découpe, on calcule, puis on passe sur le GPU.

Avant chaque cellule, essayez de prédire le résultat.

**Ce que vous allez voir**

- créer des tenseurs et lire leur forme et leur type ;
- découper, remodeler et combiner des tenseurs, sur une petite image et un tableau de ventes ;
- passer du CPU au GPU, et les deux erreurs les plus fréquentes.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [ ]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Un tenseur à partir d'une liste

**Le tenseur est la structure de données de PyTorch : un tableau de nombres, à une ou plusieurs dimensions.**

Tout passe par des tenseurs, les données comme les paramètres des modèles. Si vous connaissez les
tableaux NumPy, c'est le même principe, avec en plus le calcul sur GPU et le calcul des gradients.

Le code :
- `torch.tensor([...])` : crée un tenseur à partir d'une liste ;
- `.shape` : la **forme**, c'est-à-dire la taille de chaque dimension ;
- `.dtype` : le **type** des nombres stockés.

In [ ]:
import torch

temperatures = torch.tensor([0.5, 14.0, 15.0, 28.0, 11.0])
print(temperatures)
print(temperatures.shape, temperatures.dtype)

Cinq températures, de forme `torch.Size([5])` : une seule dimension de 5 éléments. Les nombres à virgule sont
par défaut des `float32`.

**À retenir** : `shape` et `dtype` sont les deux premières choses à vérifier sur un tenseur.

## 2. Calculer sur tout le tenseur

**Une opération s'applique à tous les éléments d'un coup, sans boucle.**

C'est plus court à écrire, et beaucoup plus rapide, surtout sur GPU, qui traite les éléments en
parallèle.

Le code :
- `temperatures * 9 / 5 + 32` : la conversion en degrés Fahrenheit, appliquée aux cinq valeurs.

Prédisez : combien de valeurs en sortie ?

In [ ]:
print(temperatures * 9 / 5 + 32)

La première température donne 32.9 degrés Fahrenheit, la plus chaude 82.4 : cinq calculs en une seule
opération, sans boucle.

**À retenir** : on écrit le calcul pour un élément, PyTorch l'applique à tous.

## 3. Une image est un tenseur

**Une image en niveaux de gris n'est qu'un tableau de pixels : 0 pour noir, 1 pour blanc.**

C'est ainsi qu'un réseau voit une image. Les réseaux convolutifs de cet après-midi travailleront sur des
tenseurs de ce type, en plus grand.

Le code :
- `torch.zeros(7, 7)` : une image 7 x 7, entièrement noire ;
- `image[3, :] = 1.0` : la ligne numéro 3 devient blanche (`:` veut dire « toutes les colonnes ») ;
- `image[:, 3] = 1.0` : la colonne numéro 3 devient blanche.

Les indices commencent à 0 : la ligne 3 est la quatrième, celle du milieu.

In [ ]:
image = torch.zeros(7, 7)
print(image)
print()
image[3, :] = 1.0
print(image)
print()
image[:, 3] = 1.0
print(image)

Un tableau 7 x 7 de zéros, avec une ligne et une colonne de 1 : la ligne 3 et la colonne 3, en comptant à
partir de 0.

**Piège fréquent** : oublier que les indices commencent à 0.

## 4. Afficher l'image

**Le même tenseur, affiché comme une image.**

Le code :
- `plt.imshow(image, cmap="gray")` : affiche un tableau 2D comme une image, en niveaux de gris.

Prédisez : quelle forme va apparaître ?

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(image, cmap="gray")
plt.show()

La croix blanche sur fond noir : le même tenseur, vu comme une image.

**À retenir** : une image n'est qu'un tenseur de pixels ; les vêtements de la démo 5 et les photos de CIFAR-10
des TP seront des tenseurs de ce type, avec trois canaux de couleur pour les photos.

## 5. Indexer : un pixel, une ligne, un bloc

**Les crochets sélectionnent une partie du tenseur.**

On découpe sans cesse les tenseurs : un exemple dans un lot, une zone d'une image, les premières
valeurs pour vérifier.

Le code :
- `image[3, 3]` : un pixel, ligne 3 et colonne 3 ;
- `image[0]` : toute la ligne 0 ;
- `image[2:5, 2:5]` : les lignes 2 à 4 et les colonnes 2 à 4 ; la borne de fin est exclue.

In [ ]:
print(image[3, 3])
print(image[0])
print(image[2:5, 2:5])

`image[3, 3]` est le pixel central, à 1 ; `image[0]` la première ligne ; `image[2:5, 2:5]` le bloc 3 x 3 du
centre, où l'on retrouve le coeur de la croix.

**Essayez** : affichez `image[:, 0]`, puis `image[::2, ::2]` (une ligne et une colonne sur deux).

## 6. Changer la forme

**On peut réorganiser les mêmes valeurs dans une autre forme.**

Chaque couche attend une forme précise. Une couche `Linear` attend un vecteur par exemple ; une
couche de convolution attend des images avec une dimension pour les canaux.

Le code :
- `image.flatten()` : aplatit l'image en une seule ligne de pixels ;
- `image.unsqueeze(0)` : ajoute une dimension de taille 1 devant.

Prédisez : combien de pixels dans l'image aplatie ?

In [ ]:
print(image.flatten().shape)
print(image.unsqueeze(0).shape)

Aplatie, l'image devient un vecteur de 49 pixels : c'est ainsi qu'un réseau de couches `Linear` lira les
images. Avec `unsqueeze(0)`, la forme devient (1, 7, 7) : un canal devant, comme les images en niveaux de gris
de torchvision.

**À retenir** : les mêmes valeurs peuvent prendre plusieurs formes ; on choisit celle qu'attend la couche
suivante.

## 7. Le type des nombres

**Chaque tenseur a un type de nombres, le `dtype`.**

Les réseaux calculent en nombres à virgule sur 32 bits (`float32`). Des entiers conviennent pour
des numéros de classe, pas pour des calculs de gradients.

Le code :
- `torch.tensor([1, 2, 3])` : des entiers donnent un tenseur d'entiers ;
- `.float()` : convertit en nombres à virgule.

In [ ]:
entiers = torch.tensor([1, 2, 3])
print(entiers.dtype)
print(entiers.float().dtype)

`int64` pour des entiers ; `float32` après conversion, le type des calculs d'un réseau.

**Piège fréquent** : certaines opérations, comme la moyenne, refusent les entiers ; on convertit d'abord avec
`.float()`.

## 8. Depuis NumPy, et retour

**PyTorch et NumPy se convertissent l'un en l'autre.**

Les données arrivent souvent sous forme de tableaux NumPy (ou de tables pandas), et les
bibliothèques de graphiques attendent des tableaux NumPy.

Le code :
- `np.array(...)` : un tableau NumPy ;
- `torch.from_numpy(tableau)` : le tenseur correspondant ;
- `.numpy()` : le chemin inverse.

In [ ]:
import numpy as np

tableau = np.array([[1.0, 2.0], [3.0, 4.0]])
tenseur = torch.from_numpy(tableau)
print(tenseur)
print(type(tenseur.numpy()))

Le tenseur garde le type du tableau NumPy, `float64` ; `.numpy()` rend un tableau NumPy. On passe de l'un à
l'autre sans effort.

**À retenir** : les réseaux calculent en `float32` ; un tableau NumPy en `float64` se convertit avec
`.float()`.

## 9. Broadcasting : combiner des formes différentes

**Le broadcasting combine un tableau et un vecteur sans écrire de boucle.**

On applique souvent une même valeur par colonne, par exemple un prix par produit ou une moyenne
par canal. PyTorch répète le vecteur sur chaque ligne, sans le copier.

Le code :
- `ventes` : 2 magasins (lignes) et 3 produits (colonnes), forme (2, 3) ;
- `prix` : un prix par produit, forme (3,) ;
- `ventes * prix` : chaque colonne est multipliée par son prix.

La règle : on compare les formes en partant de la fin ; les tailles doivent être égales, ou l'une vaut 1.

In [ ]:
ventes = torch.tensor([[10.0, 20.0, 30.0],
                       [5.0, 5.0, 5.0]])
prix = torch.tensor([2.0, 1.0, 0.5])
print(ventes * prix)

Chaque colonne est multipliée par son prix : 10 x 2 = 20, 20 x 1 = 20, 30 x 0.5 = 15 pour le premier
magasin. Le vecteur des 3 prix a été répété sur les 2 lignes, sans boucle à écrire.

**Essayez** : multipliez `ventes` par un vecteur de 2 valeurs. Que dit le message d'erreur ?

## 10. Produit matriciel

**Le produit matriciel `@` multiplie et additionne en une seule opération.**

C'est exactement le calcul d'une couche de neurones : chaque sortie est la somme des entrées
multipliées par des poids. Ici, les poids sont les prix, et la sortie le chiffre d'affaires de chaque magasin.

Le code :
- `ventes @ prix` : pour chaque magasin, la somme des quantités multipliées par les prix. Forme (2, 3) @ (3,)
  donne (2,) : une valeur par magasin.

In [ ]:
print(ventes @ prix)

55 pour le premier magasin, 17.5 pour le second : le produit matriciel multiplie puis additionne. Une
couche `Linear` fait exactement ce calcul, avec ses poids à la place des prix.

**À retenir** : (2, 3) `@` (3,) donne (2,) ; la dimension commune, ici 3, disparaît.

## 11. Réductions : résumer un tenseur

**Une réduction résume un tenseur : somme, moyenne, maximum.**

La loss est une moyenne, l'accuracy une moyenne, la prédiction d'une classe un maximum. On réduit
sans cesse.

Le code :
- `ventes.sum()` : la somme de tout ;
- `ventes.sum(dim=0)` : on additionne le long des lignes, il reste une valeur par colonne (par produit) ;
- `ventes.mean(dim=1)` : la moyenne le long des colonnes, une valeur par ligne (par magasin).

Le paramètre `dim` désigne la dimension qui disparaît.

In [ ]:
print(ventes.sum())
print(ventes.sum(dim=0))
print(ventes.mean(dim=1))

75 ventes au total ; par produit (`dim=0`, on additionne les lignes) : 15, 25 et 35 ; en moyenne par
magasin (`dim=1`) : 20 et 5.

**À retenir** : `dim` désigne la dimension qui disparaît.

## 12. Passer sur le GPU

**`.to(device)` envoie un tenseur sur le GPU ; `.cpu()` le ramène.**

Pour profiter du GPU, les données et le modèle doivent y être. On écrit le code une seule fois : il
tourne sur GPU s'il y en a un, sur CPU sinon.

Le code :
- `torch.device("cuda" if ... else "cpu")` : on choisit le device une seule fois, en début de notebook ;
- `image.to(device)` : une copie de l'image sur ce device ;
- `.device` : où se trouve un tenseur ;
- `.cpu()` : ramène un résultat sur le CPU, par exemple pour l'afficher.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_gpu = image.to(device)
print(image_gpu.device)
print((image_gpu * 2).cpu().device)

`cuda:0` : l'image est sur le GPU. Le résultat du calcul, ramené par `.cpu()`, est de nouveau sur le CPU.

**À retenir** : choisir `device` une seule fois en début de notebook, puis envoyer modèle et données avec
`.to(device)`.

## 13. Erreur : mélanger CPU et GPU

**L'erreur la plus fréquente des débutants : un calcul entre un tenseur sur CPU et un tenseur sur GPU.**

Le CPU et le GPU ont chacun leur mémoire ; PyTorch refuse de mélanger les deux sans qu'on le demande.

Le code :
- `try: ... except RuntimeError as err:` : on capture l'erreur pour afficher son message sans arrêter le
  notebook ;
- `image + image_gpu` : `image` est restée sur le CPU, `image_gpu` est sur le GPU.

Prédisez : que va dire le message ?

In [ ]:
try:
    print((image + image_gpu).device)
except RuntimeError as err:
    print("Erreur :", err)

L'erreur dit exactement le problème : deux devices dans le même calcul, `cuda:0` et `cpu`.

**Piège fréquent** : envoyer le modèle sur le GPU mais oublier les données. La correction : envoyer les deux
tenseurs sur le même device avec `.to(device)`.

## 14. Erreur : NumPy sur un tenseur GPU

**Deuxième erreur classique : convertir en NumPy un tenseur resté sur le GPU.**

NumPy ne travaille qu'avec la mémoire du CPU. C'est ce qui arrive quand on veut tracer une
prédiction faite sur GPU.

Le code :
- `image_gpu.numpy()` : la conversion directe, qui échoue sur GPU ;
- `except TypeError` : le type d'erreur levé dans ce cas.

In [ ]:
try:
    print(type(image_gpu.numpy()))
except TypeError as err:
    print("Erreur :", err)

NumPy ne sait pas lire la mémoire du GPU. Le message donne la solution : `.cpu()` d'abord, puis `.numpy()`.

**À retenir** : pour tracer ou exporter un résultat calculé sur GPU, on le ramène d'abord sur le CPU.

## À retenir de cette démo

- Un tenseur a une **forme** (`shape`), un **type** (`dtype`) et un **emplacement** (`device`, CPU ou GPU).
- Les opérations s'appliquent à tous les éléments d'un coup ; le broadcasting combine des formes compatibles ; le produit matriciel `@` est le calcul d'une couche de neurones.
- Les deux opérandes d'un calcul doivent être sur le même device ; NumPy ne lit que le CPU.